In [9]:
import numpy as np
import gymnasium as gym
import torch
import torch.nn as nn
import torch.nn.functional as F
from dataclasses import dataclass, field
from stable_baselines3 import PPO

In [10]:
class ContinuousPPO(nn.Module):
    def __init__(self, state_dim, action_dim, hidden_dim):
        super(ContinuousPPO, self).__init__()
        self.feature_extractor = nn.Sequential(
            nn.Linear(state_dim, hidden_dim),
            nn.CELU(),
        )
        self.actor_mean = nn.Sequential(
            nn.Linear(hidden_dim, action_dim),
        )
        self.actor_logstd = nn.Parameter(torch.zeros(1, action_dim))
        self.critic = nn.Linear(hidden_dim, 1)

    def forward(self, state):
        features = self.feature_extractor(state)
        action_mean = self.actor_mean(features)
        action_std = self.actor_logstd.expand_as(action_mean).exp()
        state_value = self.critic(features)  
        return action_mean, action_std, state_value

    def act(self, state):
        action_mean, action_std, state_value = self.forward(state)
        dist = torch.distributions.Normal(action_mean, action_std)
        action = dist.sample()
        return action, dist.log_prob(action).sum(dim=-1), state_value

In [11]:
@dataclass
class Trainer:
    env: gym.Env
    total_timesteps: int = 500000
    rollout_length: int = 2048
    n_epochs: int = 10
    batch_size: int = 64
    c1: float = 0.5  
    c2: float = 0.01  
    gae: float = 0.95
    gamma: float = 0.99
    clip_epsilon: float = 0.2
    learning_rate: float = 3e-4
    max_grad_norm: float = 0.5

    def __post_init__(self):
        self.model = ContinuousPPO(
            state_dim=self.env.observation_space.shape[0],
            action_dim=self.env.action_space.shape[0],
            hidden_dim=64
        )

        self.optimizer = torch.optim.Adam(self.model.parameters(), lr=self.learning_rate)
        self.buffer = []
        self.state, _ = self.env.reset()

    def _rollout(self, num_steps):
        self.buffer = []
        for _ in range(num_steps):
            state_tensor = torch.FloatTensor(self.state).unsqueeze(0)
            with torch.no_grad():
                action_tensor, log_prob_tensor, value_tensor = self.model.act(state_tensor)
            action_np = action_tensor.cpu().numpy()[0]
            # Pendulum action range is [-2, 2], Tanh output is [-1, 1]
            action_clipped = np.clip(action_np * 2.0, -2.0, 2.0)
            next_state, reward, terminated, truncated, _ = self.env.step(action_clipped)
            done = terminated or truncated
            self.buffer.append((self.state, action_np, reward, log_prob_tensor.item(), value_tensor.item(), float(done)))
            self.state = next_state
            if done: 
                self.state, _ = self.env.reset()  
                     
        next_state_tensor = torch.FloatTensor(self.state).unsqueeze(0)
        with torch.no_grad():
            _, _, next_value_tensor = self.model.act(next_state_tensor)
            next_value = 0.0 if done else next_value_tensor.item()
        return next_value

    def _compute_gae(self, next_value):
        advantages = []
        gae = 0
        for step in reversed(self.buffer):
            _, _, reward, _, value, done = step
            # delta = r + gamma * V(s') * (1-done) - V(s)
            delta = reward + self.gamma * next_value * (1 - done) - value
            gae = delta + self.gamma * self.gae * (1 - done) * gae
            advantages.insert(0, gae)
            next_value = value
        return torch.FloatTensor(advantages)

    def train(self):
        timestep = 0
        self.state, _ = self.env.reset()
        while timestep < self.total_timesteps:
            next_value = self._rollout(self.rollout_length)
            timestep += self.rollout_length
            advs = self._compute_gae(next_value)
            # Normalize advantages
            advs = (advs - advs.mean()) / (advs.std() + 1e-8)
            # Prepare batches
            s_b = torch.FloatTensor(np.array([x[0] for x in self.buffer]))
            a_b = torch.FloatTensor(np.array([x[1] for x in self.buffer]))
            lp_old_b = torch.FloatTensor([x[3] for x in self.buffer])
            v_old_b = torch.FloatTensor([x[4] for x in self.buffer])
            ret_b = advs + v_old_b
            for _ in range(self.n_epochs):
                indices = np.random.permutation(self.rollout_length)
                for start in range(0, self.rollout_length, self.batch_size):
                    idx = indices[start : start + self.batch_size]
                    # Forward pass
                    mu, std, val = self.model(s_b[idx])
                    dist = torch.distributions.Normal(mu, std)
                    lp_new = dist.log_prob(a_b[idx]).sum(dim=-1)
                    ent = dist.entropy().mean()
                    # Actor Loss (PPO Clip)
                    ratio = (lp_new - lp_old_b[idx]).exp()
                    surr1 = ratio * advs[idx]
                    surr2 = torch.clamp(ratio, 1 - self.clip_epsilon, 1 + self.clip_epsilon) * advs[idx]
                    actor_loss = -torch.min(surr1, surr2).mean()
                    # Critic Loss
                    critic_loss = F.mse_loss(val.squeeze(-1), ret_b[idx])
                    # Total Loss
                    loss = actor_loss + self.c1 * critic_loss - self.c2 * ent
                    self.optimizer.zero_grad()
                    loss.backward()
                    nn.utils.clip_grad_norm_(self.model.parameters(), self.max_grad_norm)
                    self.optimizer.step()
            if timestep % 10 * self.rollout_length == 0:
                print(f"Step: {timestep} | Loss: {loss.item():.4f}")
    def save(self, name: str):
        info = {
            "model_state_dict": self.model.state_dict(),
            "optimizer_state_dict": self.optimizer.state_dict(),
        }
        name = name.strip()
        name = name.replace(" ", "_")
        if not name.endswith(".pth"):
            name += ".pth"
        torch.save(info, name)

In [12]:
parameters = {
    "total_timesteps": 300000,
    "rollout_length": 2048,
    "n_epochs": 10,
    "batch_size": 64,
    "c1": 0.5,
    "c2": 0.01,
    "clip_epsilon": 0.2,
    "gae": 0.95,
    "gamma": 0.99,
    "learning_rate": 3e-4,
    "max_grad_norm": 0.5,
}

In [12]:
env = gym.make('Pendulum-v1')
trainer = Trainer(env=env, **parameters)
trainer.train()
trainer.save("ppo_custom_pendulum.pth")

Step: 10240 | Loss: 0.1670
Step: 20480 | Loss: 0.3175
Step: 30720 | Loss: 0.1390
Step: 40960 | Loss: 0.2526
Step: 51200 | Loss: 0.3210
Step: 61440 | Loss: 0.1425
Step: 71680 | Loss: 0.5249
Step: 81920 | Loss: 0.3748
Step: 92160 | Loss: 0.3133
Step: 102400 | Loss: 0.3158
Step: 112640 | Loss: 0.5219
Step: 122880 | Loss: 0.4096
Step: 133120 | Loss: 0.5284
Step: 143360 | Loss: 0.2526
Step: 153600 | Loss: 0.3526
Step: 163840 | Loss: 0.4210
Step: 174080 | Loss: 0.2352
Step: 184320 | Loss: 0.4363
Step: 194560 | Loss: 0.3397
Step: 204800 | Loss: 0.3403
Step: 215040 | Loss: 0.1681
Step: 225280 | Loss: 0.3597
Step: 235520 | Loss: 0.6203
Step: 245760 | Loss: 0.2638
Step: 256000 | Loss: 0.2838
Step: 266240 | Loss: 0.5993
Step: 276480 | Loss: 0.1542
Step: 286720 | Loss: 0.4946
Step: 296960 | Loss: 0.3859


In [13]:
env = gym.make('Pendulum-v1')
model = PPO("MlpPolicy", env, verbose=1, 
            learning_rate=parameters["learning_rate"], 
            gae_lambda=parameters["gae"], 
            gamma=parameters["gamma"], 
            clip_range=parameters["clip_epsilon"],
            ent_coef=parameters["c2"],
            vf_coef=parameters["c1"],
            batch_size=parameters["batch_size"],
            n_epochs=parameters["n_epochs"],
            max_grad_norm=parameters["max_grad_norm"],
            n_steps=parameters["rollout_length"]
            )
model.learn(total_timesteps=parameters["total_timesteps"])
model.save("ppo_sb3_pendulum")

Using cpu device
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.
----------------------------------
| rollout/           |           |
|    ep_len_mean     | 200       |
|    ep_rew_mean     | -1.26e+03 |
| time/              |           |
|    fps             | 2489      |
|    iterations      | 1         |
|    time_elapsed    | 0         |
|    total_timesteps | 2048      |
----------------------------------
----------------------------------------
| rollout/                |            |
|    ep_len_mean          | 200        |
|    ep_rew_mean          | -1.21e+03  |
| time/                   |            |
|    fps                  | 1761       |
|    iterations           | 2          |
|    time_elapsed         | 2          |
|    total_timesteps      | 4096       |
| train/                  |            |
|    approx_kl            | 0.00370494 |
|    clip_fraction        | 0.0283     |
|    clip_range           | 0.2        |
|    entropy_loss      

In [13]:
def test_custom(model, env, episodes=3, seed=42):
    model.eval() 
    for ep in range(episodes):
        state, _ = env.reset(seed=seed + ep if seed else None)
        episode_reward = 0
        done = False
        trunc = False
        while not (done or trunc):
            state_t = torch.FloatTensor(state).unsqueeze(0)
            with torch.no_grad():
                mu, _, _ = model(state_t)
            action = mu.cpu().numpy()[0] * 2.0 
            state, reward, done, trunc, _ = env.step(action)
            episode_reward += reward  
        print(f"Test Episode {ep + 1} | Total Reward: {episode_reward:.2f}")
    env.close()
def test_sb3(model, env, episodes=3, seed=42):
    for ep in range(episodes):
        obs = env.reset(seed=seed + ep if seed else None)[0]
        episode_reward = 0
        done = False
        trunc = False
        while not (done or trunc):
            action, _states = model.predict(obs, deterministic=True)
            obs, reward, done, trunc, _ = env.step(action)
            episode_reward += reward
            
        print(f"SB3 Test Episode {ep + 1} | Total Reward: {episode_reward:.2f}")
    env.close()

In [14]:
env_custom_ppo = gym.make('Pendulum-v1', render_mode="human")
env_custom_ppo.metadata["render_fps"] = 80
model = ContinuousPPO(
    state_dim=env_custom_ppo.observation_space.shape[0],
    action_dim=env_custom_ppo.action_space.shape[0],
    hidden_dim=64
)
info = torch.load("ppo_custom_pendulum.pth")
model.load_state_dict(info["model_state_dict"])
test_custom(model, env_custom_ppo, episodes=3, seed=42)

Test Episode 1 | Total Reward: -130.30
Test Episode 2 | Total Reward: -131.81
Test Episode 3 | Total Reward: -257.47


In [15]:
env = gym.make('Pendulum-v1', render_mode="human")
env.metadata["render_fps"] = 80
model_sb3 = PPO.load("ppo_sb3_pendulum")
test_sb3(model_sb3, env, episodes=3, seed=42)

SB3 Test Episode 1 | Total Reward: -129.83
SB3 Test Episode 2 | Total Reward: -128.98
SB3 Test Episode 3 | Total Reward: -246.95
